# 14 — Paper Figures: Shared Style and Class Imbalance

Builds the shared plotting style and result loader used by notebooks 15-18,
and produces a class-imbalance-across-splits figure (`dataset_splits.pdf/png`).
The dataset-construction schematic (`dataset_overview.pdf/png`) and the
per-original-label distribution live in notebook 19 instead.

**Style source:** the colors, fonts and chart grammar here are not a new
design — they are copied from `../SEP_DataAugmentation 2/paper/figures/`,
sampled directly from that project's own figure-generation code
(`8_Figures.ipynb`). Serif type, the Okabe-Ito colorblind-safe accent
palette, a single highlighted "answer" per panel, raw run-level dots
instead of box-and-whisker chrome, and one clear finding per panel.

**Reads:** `./results/*.txt` (read-only, same files notebooks 11-13 use).
**Writes:** `./paper/figures/*.pdf` and `*.png`.
**Does not touch:** notebooks 11, 12, 13, or any of their output — this is
a parallel, independent figure set, not a replacement.

## 1. Shared Style and Loader

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

RESULTS_DIR = "./results"
FIG_DIR = "./paper/Figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ══════════════════════════════════════════════════════════════
# STYLE — copied from SEP_DataAugmentation 2's own figure code
# (../SEP_DataAugmentation 2/8_Figures.ipynb, cell 2 "Shared Style")
# ══════════════════════════════════════════════════════════════

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.4, "ytick.major.size": 2.4,
    "xtick.labelsize": 6.4, "ytick.labelsize": 6.4,
    "axes.labelsize": 7.0, "axes.titlesize": 7.4, "legend.fontsize": 6.2,
    "axes.edgecolor": "0.35", "text.color": "0.0",
    "axes.labelcolor": "0.0", "xtick.color": "0.25", "ytick.color": "0.25",
})

# Okabe-Ito colorblind-safe palette. TEAL marks "the answer" throughout;
# VERM marks a second series or an actively harmful result; GOLD/BLUE are
# spare accents for when a third category is needed.
BLUE, VERM, TEAL, GOLD = "#0072B2", "#D55E00", "#009E73", "#CC9A00"
GREY, LG = "0.45", "#DCDCDC"

CLASSIFIERS = ["gru", "patchtst", "svm", "inceptiontime"]
CLASSIFIER_LABELS = {"gru": "GRU", "patchtst": "PatchTST",
                     "svm": "SVM", "inceptiontime": "InceptionTime"}

COLUMN = {"TP": 0, "TN": 1, "FP": 2, "FN": 3, "tss": 4, "hss1": 5, "hss2": 6,
         "gss": 7, "recall": 8, "f1": 9, "accuracy": 10, "train_time": 11,
         "infer_time": 12, "far": 13, "bias": 14}


def finish(ax, ylab=None, xlab=None, grid="y"):
    """Strip top/right spines, add a light one-axis grid, keep it below the data.

    grid=None skips the grid entirely (e.g. a filled-contour background,
    where gridlines would just add clutter on top of it).
    """
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if grid is not None:
        ax.grid(True, axis=grid, color="0.88", lw=0.5, zorder=0)
    ax.set_axisbelow(True)
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)


def load_col(clf, key, col=COLUMN["tss"], results_dir=RESULTS_DIR):
    """One metric column from one result file; one value per run."""
    fp = os.path.join(results_dir, f"{clf}_{key}.txt")
    if not os.path.exists(fp):
        return np.array([])
    return np.array([float(l.split(",")[col]) for l in open(fp) if l.strip()])


def load_all_classifiers(key, col=COLUMN["tss"], classifiers=CLASSIFIERS,
                         results_dir=RESULTS_DIR):
    """Pool one metric column across all classifiers for one result key."""
    out = []
    for clf in classifiers:
        out.extend(load_col(clf, key, col, results_dir).tolist())
    return np.array(out)


print("Style + loader ready.")
print("Results dir :", os.path.abspath(RESULTS_DIR))
print("Figures out :", os.path.abspath(FIG_DIR))


Style + loader ready.
Results dir : /Users/samskanderi/Documents/Claude/SEP_DataAugmentation/results
Figures out : /Users/samskanderi/Documents/Claude/SEP_DataAugmentation/paper/Figures


## Figure — Dataset and Class Imbalance

In [2]:
# FIG -- dataset overview: class imbalance across splits
#
# Train counts are the Tomek-cleaned training split (final_split_data_
# HybridNorm_Tomek); validation and test are untouched by any cleaning or
# resampling step, so their counts are the same throughout the project.
# The split .pkl files are removed by each notebook's own cleanup cell
# once it is no longer needed, so these counts are recorded here rather
# than recomputed -- they were read directly from the split files earlier
# in the project (train: notebook 8's ADASYN margin check; val/test:
# notebook 4's original split report) and cross-checked against the
# TP+FN / FP+TN totals in every result file's test row.

fig, ax = plt.subplots(figsize=(3.4, 2.6))

splits = ["Train", "Validation", "Test"]
nonsep = [12323, 1763, 3525]
sep    = [118, 17, 34]

x = np.arange(len(splits))
w = 0.32
ax.bar(x - w / 2, nonsep, w, color="0.62", edgecolor="none", zorder=3)
ax.bar(x + w / 2, sep, w, color=TEAL, edgecolor="none", zorder=3)

for xi, n, s in zip(x, nonsep, sep):
    ax.text(xi - w / 2, n * 1.18, f"{n:,}", ha="center", fontsize=6.0, color="0.0")
    ax.text(xi + w / 2, s * 1.9, f"{s}\n({100 * s / (s + n):.2f}%)",
            ha="center", fontsize=6.0, color=TEAL, linespacing=1.3)

ax.set_yscale("log")
ax.set_ylim(5, 4e4)
ax.set_xticks(x)
ax.set_xticklabels(splits, fontsize=6.4)
finish(ax, ylab="Samples (log scale)")
ax.set_title("Class Imbalance Across Splits", loc="left", fontsize=7.4, pad=6)

legend_handles = [Patch(facecolor="0.62", label="Non-SEP"),
                  Patch(facecolor=TEAL, label="SEP")]
ax.legend(handles=legend_handles, frameon=False, loc="upper right",
          handlelength=1.0, handletextpad=0.4, borderpad=0.1,
          labelspacing=0.25, fontsize=6.2)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/dataset_splits.pdf")
fig.savefig(f"{FIG_DIR}/dataset_splits.png", dpi=300)
plt.show()
print("Saved dataset_splits.pdf/png")
print("SEP events are under 1% of samples in every split "
      f"({[round(100*s/(s+n),2) for s,n in zip(sep,nonsep)]}%).")


Saved dataset_splits.pdf/png
SEP events are under 1% of samples in every split ([0.95, 0.96, 0.96]%).


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_41597/1240359089.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
